In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, 
                             accuracy_score, precision_score, recall_score, f1_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# ==========================================
# 0. Environment Setup
# ==========================================
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")

SEED = 0
np.random.seed(SEED)

# ==========================================
# 1. Data Preparation
# ==========================================
print("Loading Stage 1 data...")
# Load pre-ICU / admission data
df_before = pd.read_csv('全部病房患者/内部训练集/preicu.csv')
cols_stage1 = [
    'gender', 'age', 'admissionroute', 'vasopressor', 'MV',
    'temp_max', 'temp_min', 'resp_max', 'resp_min', 'hr_max', 'hr_min', 'map_max', 'map_min', 
    'tbil', 'glu', 'cre', 'plt', 'wbc', 'lym'
]

X = df_before[cols_stage1]
y = df_before['death_90d'].values

print(f"Number of samples: {len(X)} | Number of features: {len(cols_stage1)}")
print(f"Positive sample ratio: {np.mean(y):.2%}")

# ==========================================
# 2. 5-Fold Cross Validation (Logistic Regression)
# ==========================================
print("--- Starting 5-Fold CV (Stage 1 Logistic Regression) ---")
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

cv_metrics = {'auc': [], 'acc': [], 'prec': [], 'recall': [], 'f1': []}
tprs = []
mean_fpr = np.linspace(0, 1, 100)
fold_no = 1

for train_idx, val_idx in kfold.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    
    # Scaling
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    
    # Model Training
    model = LogisticRegression(class_weight='balanced', solver='liblinear', random_state=SEED)
    model.fit(X_train_scaled, y_train)
    y_pred_prob = model.predict_proba(X_val_scaled)[:, 1]
    
    # Evaluation Metrics
    fpr, tpr, _ = roc_curve(y_val, y_pred_prob)
    roc_auc = auc(fpr, tpr)
    cv_metrics['auc'].append(roc_auc)
    
    tprs.append(np.interp(mean_fpr, fpr, tpr))
    tprs[-1][0] = 0.0
    
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_prob)
    with np.errstate(divide='ignore', invalid='ignore'):
        f1_scores_curve = 2 * (precisions * recalls) / (precisions + recalls)
    f1_scores_curve = np.nan_to_num(f1_scores_curve)
    
    best_idx = np.argmax(f1_scores_curve)
    best_f1 = f1_scores_curve[best_idx]
    best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    y_pred_class = (y_pred_prob >= best_thresh).astype(int)
    
    acc = accuracy_score(y_val, y_pred_class)
    prec = precision_score(y_val, y_pred_class, zero_division=0)
    rec = recall_score(y_val, y_pred_class, zero_division=0)
    
    cv_metrics['acc'].append(acc)
    cv_metrics['prec'].append(prec)
    cv_metrics['recall'].append(rec)
    cv_metrics['f1'].append(best_f1)
    
    print(f"Fold {fold_no} -> AUC: {roc_auc:.4f} | F1: {best_f1:.4f} | Acc: {acc:.4f} (Thresh: {best_thresh:.3f})")
    fold_no += 1

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, 
                             accuracy_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# ==========================================
# 0. Environment Setup
# ==========================================
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")

SEED = 0
np.random.seed(SEED)

# ==========================================
# 1. Data Preparation
# ==========================================
print("Loading data...")
df_before = pd.read_csv('全部病房患者/内部训练集/preicu.csv')
df_after = pd.read_csv('全部病房患者/内部训练集/icu.csv') 

cols_stage1 = [
    'gender', 'age', 'admissionroute', 'vasopressor', 'MV',
    'temp_max', 'temp_min', 'resp_max', 'resp_min', 'hr_max', 'hr_min', 'map_max', 'map_min', 
    'tbil', 'glu', 'cre', 'plt', 'wbc', 'lym'
]

cols_stage2_raw = [
    'gender', 'age', 'admissionroute', 'vasopressor', 'MV',
    'temp_max', 'temp_min', 'resp_max', 'resp_min', 'hr_max', 'hr_min', 'map_max', 'map_min', 
    'glu', 'tbil', 'cre', 'spo2', 'plt', 'wbc', 'lym', 'ph', 'lac'
]

binary_cols = ['gender', 'vasopressor', 'MV', 'admissionroute']
continuous_cols = [c for c in cols_stage2_raw if c not in binary_cols]

y_all = df_before['death_90d'].values
indices = np.arange(len(df_before))

def optimize_model(estimator, param_dist, name, X, y):
    """Perform hyperparameter optimization using RandomizedSearchCV."""
    print(f"    Optimizing {name} ...")
    search = RandomizedSearchCV(
        estimator, param_distributions=param_dist, n_iter=100,
        scoring='roc_auc', cv=5, n_jobs=-1, random_state=SEED, verbose=0
    )
    search.fit(X, y)
    return search.best_estimator_

# ==========================================
# 2. 5-Fold Cross Validation (Stacking Ensemble)
# ==========================================
print("\n--- Starting 5-Fold CV (Stacking Ensemble with Hyperparameter Tuning) ---")
print("Base models: XGBoost, LightGBM, Random Forest, GaussianNB, SVM") 
print("Meta-model: Logistic Regression")

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_metrics = {'auc': [], 'acc': [], 'prec': [], 'recall': [], 'f1': []}
tprs = []
mean_fpr = np.linspace(0, 1, 100)
fold_no = 1

for train_idx, val_idx in kfold.split(indices, y_all):
    print(f"\n========== Fold {fold_no} ==========")

    # A. Stage 1: LR generates risk score
    X_s1_train = df_before.iloc[train_idx][cols_stage1]
    y_train = y_all[train_idx]
    X_s1_val = df_before.iloc[val_idx][cols_stage1]
    y_val = y_all[val_idx]
    
    scaler_s1 = RobustScaler()
    X_s1_train_scaled = scaler_s1.fit_transform(X_s1_train)
    X_s1_val_scaled = scaler_s1.transform(X_s1_val)
    
    lr_s1 = LogisticRegression(class_weight='balanced', solver='liblinear', random_state=SEED)
    lr_s1.fit(X_s1_train_scaled, y_train)
    
    risk_train = lr_s1.predict_proba(X_s1_train_scaled)[:, 1]
    risk_val = lr_s1.predict_proba(X_s1_val_scaled)[:, 1]

    # B. Stage 2: Data Preprocessing
    X_s2_train_raw = df_after.iloc[train_idx][cols_stage2_raw].copy()
    X_s2_val_raw = df_after.iloc[val_idx][cols_stage2_raw].copy()
    
    X_s2_train_raw['pre_icu_risk'] = risk_train
    X_s2_val_raw['pre_icu_risk'] = risk_val
    
    current_cont_cols = continuous_cols + ['pre_icu_risk']
    preprocessor = ColumnTransformer(
        transformers=[
            ('cont', RobustScaler(), current_cont_cols),
            ('cat', 'passthrough', binary_cols)
        ],
        verbose_feature_names_out=False
    )
    preprocessor.set_output(transform='pandas')
    
    X_train_final = preprocessor.fit_transform(X_s2_train_raw)
    X_val_final = preprocessor.transform(X_s2_val_raw)

    # C. Hyperparameter Optimization (RandomizedSearchCV)
    print("  --- Hyperparameter Optimization ---")
    pos_count = np.sum(y_train == 1)
    neg_count = np.sum(y_train == 0)
    scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0
    
    # Define parameter space
    xgb_params = {
        'n_estimators': [300, 500, 700],
        'learning_rate': [0.01],
        'max_depth': [3, 4, 5],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9],
        'min_child_weight': [2, 3, 4],
        'scale_pos_weight': [scale_pos_weight]
    }
    lgb_params = {
        'n_estimators': [300, 500, 700],
        'num_leaves': [15, 31],
        'learning_rate': [0.01],
        'max_depth': [4, 5, 6],
        'subsample': [0.7, 0.8, 0.9],
        'min_child_samples': [10, 20],
        'scale_pos_weight': [scale_pos_weight]
    }
    rf_params = {
        'n_estimators': [300, 500, 700],
        'max_depth': [5, 10],
        'min_samples_split': [3, 4, 5],
        'min_samples_leaf': [2, 3, 4],
        'max_features': ['sqrt'],
        'class_weight': ['balanced']
    }
    svm_params = {
        'C': [0.5, 1.0, 2.0],
        'kernel': ['rbf', 'linear'],
        'gamma': ['scale', 'auto']
    }
    
    # Optimize sequentially to find the best models for the current fold
    best_xgb = optimize_model(xgb.XGBClassifier(eval_metric='auc', random_state=SEED, n_jobs=1), 
                              xgb_params, "XGBoost", X_train_final, y_train)
    best_lgbm = optimize_model(lgb.LGBMClassifier(objective='binary', metric='auc', verbosity=-1, random_state=SEED, n_jobs=1), 
                               lgb_params, "LightGBM", X_train_final, y_train)
    best_rf = optimize_model(RandomForestClassifier(random_state=SEED, n_jobs=-1), 
                             rf_params, "Random Forest", X_train_final, y_train)
    best_svm = optimize_model(SVC(probability=True, random_state=SEED), 
                              svm_params, "SVM", X_train_final, y_train)
    best_gnb = GaussianNB(var_smoothing=1e-9)

    # D. Train Final Stacking Model
    print("  --- Training Final Stacking Model ---")
    meta_learner = LogisticRegression(C=1.0, random_state=SEED)
    stacking_model = StackingClassifier(
        estimators=[
            ('xgb', best_xgb),
            ('lgbm', best_lgbm),
            ('rf', best_rf),
            ('svm', best_svm),
            ('gnb', best_gnb)
        ],
        final_estimator=meta_learner,
        cv=5, 
        n_jobs=-1,
        passthrough=False 
    )
    stacking_model.fit(X_train_final, y_train)

    # E. Evaluate current fold performance
    y_pred_prob = stacking_model.predict_proba(X_val_final)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, y_pred_prob)
    roc_auc = auc(fpr, tpr)
    cv_metrics['auc'].append(roc_auc)
    
    tprs.append(np.interp(mean_fpr, fpr, tpr))
    tprs[-1][0] = 0.0
    
    precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_prob)
    with np.errstate(divide='ignore', invalid='ignore'):
        f1_scores_curve = 2 * (precisions * recalls) / (precisions + recalls)
    f1_scores_curve = np.nan_to_num(f1_scores_curve)
    
    best_idx = np.argmax(f1_scores_curve)
    best_f1 = f1_scores_curve[best_idx]
    best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
    y_pred_class = (y_pred_prob >= best_thresh).astype(int)
    
    acc = accuracy_score(y_val, y_pred_class)
    prec = precision_score(y_val, y_pred_class, zero_division=0)
    rec = recall_score(y_val, y_pred_class, zero_division=0)
    
    cv_metrics['acc'].append(acc)
    cv_metrics['prec'].append(prec)
    cv_metrics['recall'].append(rec)
    cv_metrics['f1'].append(best_f1)
    
    print(f"  --> Fold {fold_no} Result: AUC: {roc_auc:.4f} | F1: {best_f1:.4f} | Acc: {acc:.4f}")
    fold_no += 1

In [ ]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, 
                              StackingClassifier)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, accuracy_score, precision_score, recall_score)
import xgboost as xgb
import lightgbm as lgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization, Activation
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import tensorflow.keras.backend as K

# ==========================================
# 0. Environment Setup
# ==========================================
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")

SEED = 0
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ==========================================
# 1. Core Model Definition Factory Functions
# ==========================================
def build_keras_mlp(input_dim):
    """Build Keras MLP model architecture"""
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(32, kernel_regularizer=l2(0.005)),
        BatchNormalization(),
        Activation('swish'),
        Dropout(0.3),
        Dense(32, kernel_regularizer=l2(0.005)),
        BatchNormalization(),
        Activation('swish'),
        Dropout(0.3),
        Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer=Adam(learning_rate=0.001), 
                  loss='binary_crossentropy', 
                  metrics=[tf.keras.metrics.AUC(name='auc')])
    return model

def get_models(scale_pos_weight, input_dim):
    """
    Dynamically generate new model instances for each fold.
    Returns a dictionary: Key=model name, Value=model object.
    """
    models = {}
    
    # 1. XGBoost
    models['XGBoost'] = xgb.XGBClassifier(
        n_estimators=300, learning_rate=0.01, max_depth=3,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        scale_pos_weight=scale_pos_weight, eval_metric='auc',
        random_state=SEED, n_jobs=-1
    )
    
    # 2. LightGBM
    models['LightGBM'] = lgb.LGBMClassifier(
        n_estimators=300, learning_rate=0.01, num_leaves=15, max_depth=4,
        subsample=0.8, colsample_bytree=0.8, min_child_samples=20,
        scale_pos_weight=scale_pos_weight, objective='binary',
        random_state=SEED, n_jobs=-1, verbosity=-1
    )
    
    # 3. Random Forest
    models['Random Forest'] = RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_split=5, min_samples_leaf=4,
        max_features='sqrt', class_weight='balanced',
        random_state=SEED, n_jobs=-1
    )
    
    # 4. GBDT
    models['GBDT'] = GradientBoostingClassifier(
        n_estimators=300, learning_rate=0.01, max_depth=5,
        subsample=0.8, min_samples_split=5, min_samples_leaf=5,
        random_state=SEED
    )
    
    # 5. Gaussian Naive Bayes
    models['GaussianNB'] = GaussianNB(var_smoothing=1e-8)
    
    # 6. SVM
    models['SVM'] = SVC(
        C=0.5, kernel='rbf', gamma='scale', class_weight='balanced',
        probability=True, random_state=SEED
    )
    
    # 7. MLP (Keras)
    models['MLP'] = build_keras_mlp(input_dim)
    
    # 8. Stacking (using the above base models)
    base_estimators = [
        ('xgb', models['XGBoost']),
        ('lgbm', models['LightGBM']),
        ('rf', models['Random Forest']),
        ('svm', models['SVM']),
        ('gnb', models['GaussianNB'])
    ]
    models['Stacking'] = StackingClassifier(
        estimators=base_estimators,
        final_estimator=LogisticRegression(class_weight='balanced', random_state=SEED),
        cv=5, n_jobs=-1, passthrough=False
    )
    
    return models

# ==========================================
# 2. Data Preparation
# ==========================================
print("Loading data...")
df_before = pd.read_csv('全部病房患者/内部训练集/preicu.csv')
df_after = pd.read_csv('全部病房患者/内部训练集/icu.csv')

cols_stage1 = [
    'gender', 'age', 'admissionroute', 'vasopressor', 'MV',
    'temp_max', 'temp_min', 'resp_max', 'resp_min', 'hr_max', 'hr_min', 'map_max', 'map_min',
    'tbil', 'glu', 'cre', 'plt', 'wbc', 'lym'
]
cols_stage2_raw = [
    'gender', 'age', 'admissionroute', 'vasopressor', 'MV',
    'temp_max', 'temp_min', 'resp_max', 'resp_min', 'hr_max', 'hr_min', 'map_max', 'map_min',
    'glu', 'tbil', 'cre', 'spo2', 'plt', 'wbc', 'lym', 'ph', 'lac'
]

binary_cols = ['gender', 'vasopressor', 'MV', 'admissionroute']
continuous_cols = [c for c in cols_stage2_raw if c not in binary_cols]
y_all = df_before['death_90d'].values
indices = np.arange(len(df_before))

# ==========================================
# 3. Core 5-Fold Cross Validation Loop
# ==========================================
print("\n--- Executing Integrated Multi-model 5-Fold CV ---")
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
model_names = ['XGBoost', 'LightGBM', 'Random Forest', 'GBDT', 'GaussianNB', 'SVM', 'MLP', 'Stacking']
cv_results = {name: {'auc': [], 'acc': [], 'prec': [], 'recall': [], 'f1': [], 'tprs': []} for name in model_names}

mean_fpr = np.linspace(0, 1, 100)
fold_no = 1

for train_idx, val_idx in kfold.split(indices, y_all):
    print(f"\n========== Fold {fold_no} ==========")
    K.clear_session()

    X_s1_train, X_s1_val = df_before.iloc[train_idx][cols_stage1], df_before.iloc[val_idx][cols_stage1]
    y_train, y_val = y_all[train_idx], y_all[val_idx]
    
    scaler_s1 = RobustScaler()
    X_s1_train_scaled = scaler_s1.fit_transform(X_s1_train)
    X_s1_val_scaled = scaler_s1.transform(X_s1_val)
    
    lr_s1 = LogisticRegression(class_weight='balanced', solver='liblinear', random_state=SEED)
    lr_s1.fit(X_s1_train_scaled, y_train)
    risk_train = lr_s1.predict_proba(X_s1_train_scaled)[:, 1]
    risk_val = lr_s1.predict_proba(X_s1_val_scaled)[:, 1]

    X_s2_train_raw = df_after.iloc[train_idx][cols_stage2_raw].copy()
    X_s2_val_raw = df_after.iloc[val_idx][cols_stage2_raw].copy()
    X_s2_train_raw['pre_icu_risk'] = risk_train
    X_s2_val_raw['pre_icu_risk'] = risk_val
    
    current_cont_cols = continuous_cols + ['pre_icu_risk']
    preprocessor = ColumnTransformer(
        transformers=[
            ('cont', RobustScaler(), current_cont_cols),
            ('cat', 'passthrough', binary_cols)
        ],
        verbose_feature_names_out=False
    )
    
    X_train_final = preprocessor.fit_transform(X_s2_train_raw)
    X_val_final = preprocessor.transform(X_s2_val_raw)
    
    pos_count, neg_count = np.sum(y_train == 1), np.sum(y_train == 0)
    scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0

    models_dict = get_models(scale_pos_weight, input_dim=X_train_final.shape[1])
    
    for model_name in model_names:
        print(f"  Training {model_name}...")
        model = models_dict[model_name]
        
        if model_name == 'MLP':
            class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
            cw_dict = dict(enumerate(class_weights))
            callbacks = [
                EarlyStopping(monitor='val_auc', patience=20, restore_best_weights=True, mode='max', verbose=0),
                ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=10, min_lr=1e-5, mode='max', verbose=0)
            ]
            model.fit(X_train_final, y_train, validation_data=(X_val_final, y_val),
                      epochs=150, batch_size=32, callbacks=callbacks, class_weight=cw_dict, verbose=0)
            y_pred_prob = model.predict(X_val_final, verbose=0).ravel()
            
        elif model_name == 'XGBoost':
            model.fit(X_train_final, y_train, eval_set=[(X_val_final, y_val)], verbose=False)
            y_pred_prob = model.predict_proba(X_val_final)[:, 1]
            
        elif model_name == 'LightGBM':
            model.fit(X_train_final, y_train, eval_set=[(X_val_final, y_all[val_idx])], eval_metric='auc')
            y_pred_prob = model.predict_proba(X_val_final)[:, 1]
            
        else:
            model.fit(X_train_final, y_train)
            y_pred_prob = model.predict_proba(X_val_final)[:, 1]
            
        fpr, tpr, _ = roc_curve(y_val, y_pred_prob)
        roc_auc = auc(fpr, tpr)
        
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        cv_results[model_name]['tprs'].append(interp_tpr)
        cv_results[model_name]['auc'].append(roc_auc)
        
        precisions, recalls, thresholds = precision_recall_curve(y_val, y_pred_prob)
        with np.errstate(divide='ignore', invalid='ignore'):
            f1_scores = np.nan_to_num(2 * (precisions * recalls) / (precisions + recalls))
            
        best_idx = np.argmax(f1_scores)
        best_f1 = f1_scores[best_idx]
        best_thresh = thresholds[best_idx] if best_idx < len(thresholds) else 0.5
        y_pred_class = (y_pred_prob >= best_thresh).astype(int)
        
        acc = accuracy_score(y_val, y_pred_class)
        prec = precision_score(y_val, y_pred_class, zero_division=0)
        rec = recall_score(y_val, y_pred_class, zero_division=0)
        
        cv_results[model_name]['f1'].append(best_f1)
        cv_results[model_name]['acc'].append(acc)
        cv_results[model_name]['prec'].append(prec)
        cv_results[model_name]['recall'].append(rec)

    fold_no += 1

# ==========================================
# 4. Metrics Summary and Printing
# ==========================================
print("\n" + "="*80)
print(f"{'Model':<15} | {'AUC (Mean ± Std)':<18} | {'F1 (Mean ± Std)':<18} | {'Acc (Mean ± Std)':<18} | {'Prec (Mean ± Std)':<18} | {'Recall (Mean ± Std)':<18}")
print("="*80)

final_stats = {}
for name in model_names:
    stats = {}
    for metric in ['auc', 'f1', 'acc', 'prec', 'recall']:
        mean_val = np.mean(cv_results[name][metric])
        std_val = np.std(cv_results[name][metric])
        stats[f'{metric}_mean'] = mean_val
        stats[f'{metric}_std'] = std_val
    final_stats[name] = stats
    
    print(f"{name:<15} | {stats['auc_mean']:.3f} ± {stats['auc_std']:.3f} | {stats['f1_mean']:.3f} ± {stats['f1_std']:.3f} | {stats['acc_mean']:.3f} ± {stats['acc_std']:.3f} | {stats['prec_mean']:.3f} ± {stats['prec_std']:.3f} | {stats['recall_mean']:.3f} ± {stats['recall_std']:.3f}")
print("="*80)

# ==========================================
# 5. Plot Mean ROC Curves for all models
# ==========================================
plt.figure(figsize=(10, 9), dpi=150)
colors = {
    'XGBoost': '#1f77b4',
    'LightGBM': '#ff7f0e',
    'Random Forest': '#2ca02c',
    'GBDT': '#d62728',
    'GaussianNB': '#9467bd',
    'SVM': '#8c564b',
    'MLP': '#e377c2',
    'Stacking': '#17becf'
}

for name in model_names:
    mean_tpr = np.mean(cv_results[name]['tprs'], axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = final_stats[name]['auc_mean']
    std_auc = final_stats[name]['auc_std']
    
    plt.plot(mean_fpr, mean_tpr, color=colors[name], lw=2.5, alpha=0.9,
             label=f'{name} (AUC = {mean_auc:.3f} ± {std_auc:.3f})')
             
plt.plot([0, 1], [0, 1], linestyle='--', lw=2, color='gray', alpha=0.8, label='Chance')
plt.xlim([-0.02, 1.02])
plt.ylim([-0.02, 1.02])
plt.xlabel('False Positive Rate', fontsize=14, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=14, fontweight='bold')
plt.title('Comparison of Mean ROC Curves (Two-Stage Models)', fontsize=16, fontweight='bold', pad=20)
plt.legend(loc="lower right", fontsize=11, frameon=True, fancybox=True, edgecolor='lightgray')
plt.grid(True, linestyle=':', linewidth=0.6, alpha=0.8)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.utils import resample
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, 
                             accuracy_score, precision_score, recall_score, f1_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

# ==========================================
# 0. Environment Setup
# ==========================================
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")

SEED = 0
np.random.seed(SEED)

# ==========================================
# 1. Data Preparation
# ==========================================
print("Loading data...")
df_before_int = pd.read_csv('全部病房患者/内部训练集/preicu.csv')
df_after_int = pd.read_csv('全部病房患者/内部训练集/icu.csv')
df_before_ext = pd.read_csv('全部病房患者/外部测试集/preicu.csv')
df_after_ext = pd.read_csv('全部病房患者/外部测试集/icu.csv')

cols_stage1 = [
    'gender', 'age', 'admissionroute', 'vasopressor', 'MV',
    'temp_max', 'temp_min', 'resp_max', 'resp_min', 'hr_max', 'hr_min', 'map_max', 'map_min',
    'tbil', 'glu', 'cre', 'plt', 'wbc', 'lym'   
]
cols_stage2_raw = [
    'gender', 'age', 'admissionroute', 'vasopressor', 'MV',
    'temp_max', 'temp_min', 'resp_max', 'resp_min', 'hr_max', 'hr_min', 'map_max', 'map_min',
    'glu', 'tbil', 'cre', 'spo2', 'plt', 'wbc', 'lym', 'ph', 'lac'
]

binary_cols = ['gender', 'vasopressor', 'MV', 'admissionroute']
continuous_cols_base = [c for c in cols_stage2_raw if c not in binary_cols] 

y_int = df_before_int['death_90d'].values
y_ext = df_before_ext['death_90d'].values

# ==========================================
# 2. Dataset Splitting & Stage 1 Risk Calculation
# ==========================================
indices_int = np.arange(len(df_before_int))
train_idx, val_idx = train_test_split(indices_int, test_size=0.2, stratify=y_int, random_state=SEED)

y_train = y_int[train_idx]
y_val = y_int[val_idx]

pos_count = np.sum(y_train == 1)
neg_count = np.sum(y_train == 0)
scale_pos_weight = neg_count / pos_count if pos_count > 0 else 1.0

X_s1_train = df_before_int.iloc[train_idx][cols_stage1].reset_index(drop=True)
X_s1_val = df_before_int.iloc[val_idx][cols_stage1].reset_index(drop=True)
X_s1_ext = df_before_ext[cols_stage1].reset_index(drop=True)

scaler_s1 = RobustScaler()
X_s1_train_scaled = scaler_s1.fit_transform(X_s1_train)
X_s1_val_scaled = scaler_s1.transform(X_s1_val)
X_s1_ext_scaled = scaler_s1.transform(X_s1_ext)

lr_s1 = LogisticRegression(class_weight='balanced', solver='liblinear', random_state=SEED)
lr_s1.fit(X_s1_train_scaled, y_train)

risk_train = lr_s1.predict_proba(X_s1_train_scaled)[:, 1]
risk_val = lr_s1.predict_proba(X_s1_val_scaled)[:, 1]
risk_ext = lr_s1.predict_proba(X_s1_ext_scaled)[:, 1]

# Stage 2 base data
X_s2_train_base = df_after_int.iloc[train_idx][cols_stage2_raw].reset_index(drop=True)
X_s2_val_base = df_after_int.iloc[val_idx][cols_stage2_raw].reset_index(drop=True)
X_s2_ext_base = df_after_ext[cols_stage2_raw].reset_index(drop=True)

# ==========================================
# 3. Define Bootstrap Evaluation Function
# ==========================================
def bootstrap_metrics_ci_with_roc(model, X, y, threshold, n_bootstraps=1000):
    """Calculate Bootstrap Confidence Intervals and ROC metrics."""
    y_prob_full = model.predict_proba(X)[:, 1]
    boot_stats = {'AUC': [], 'Accuracy': [], 'Precision': [], 'Recall': [], 'F1': []}
    mean_fpr = np.linspace(0, 1, 100)
    tprs = []
    
    for i in range(n_bootstraps):
        indices = resample(np.arange(len(y)), random_state=i)
        y_true_boot = y[indices]
        y_prob_boot = y_prob_full[indices]
        
        if len(np.unique(y_true_boot)) < 2:
            continue
            
        fpr, tpr, _ = roc_curve(y_true_boot, y_prob_boot)
        boot_stats['AUC'].append(auc(fpr, tpr))
        
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        
        y_pred_boot = (y_prob_boot >= threshold).astype(int)
        boot_stats['Accuracy'].append(accuracy_score(y_true_boot, y_pred_boot))
        boot_stats['Precision'].append(precision_score(y_true_boot, y_pred_boot, zero_division=0))
        boot_stats['Recall'].append(recall_score(y_true_boot, y_pred_boot, zero_division=0))
        boot_stats['F1'].append(f1_score(y_true_boot, y_pred_boot, zero_division=0))
        
    results_str = {}
    for metric, values in boot_stats.items():
        mean_val = np.mean(values)
        ci_lower = np.percentile(values, 2.5)
        ci_upper = np.percentile(values, 97.5)
        results_str[metric] = f"{mean_val:.3f} ({ci_lower:.3f}-{ci_upper:.3f})"
        
    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    tpr_lower = np.percentile(tprs, 2.5, axis=0)
    tpr_upper = np.percentile(tprs, 97.5, axis=0)
    
    return results_str, mean_fpr, mean_tpr, tpr_lower, tpr_upper

# ==========================================
# 4. Main Loop for Strategy Comparison (ICU_Only vs Two_Stage)
# ==========================================
strategies = ['ICU_Only', 'Two_Stage']
all_results = {}

print("\n=== Model Training and Evaluation (Bootstrap CI) ===")

for strategy in strategies:
    print(f">> Strategy: {strategy}")
    
    X_train_raw, X_val_raw, X_ext_raw = X_s2_train_base.copy(), X_s2_val_base.copy(), X_s2_ext_base.copy()
    
    if strategy == 'Two_Stage':
        X_train_raw['pre_icu_risk'], X_val_raw['pre_icu_risk'], X_ext_raw['pre_icu_risk'] = risk_train, risk_val, risk_ext
        current_continuous = continuous_cols_base + ['pre_icu_risk']
    else:
        current_continuous = continuous_cols_base.copy()
        
    preprocessor = ColumnTransformer(
        transformers=[('cont', RobustScaler(), current_continuous), ('cat', 'passthrough', binary_cols)],
        verbose_feature_names_out=False
    )
    
    X_train_final = preprocessor.fit_transform(X_train_raw)
    X_val_final = preprocessor.transform(X_val_raw)
    X_ext_final = preprocessor.transform(X_ext_raw)
    
    # Hyperparameter Optimization and Model Training
    xgb_params = {'n_estimators': [300, 500, 700], 'learning_rate': [0.01], 'max_depth': [3, 4, 5], 'subsample': [0.7, 0.8, 0.9], 'colsample_bytree': [0.7, 0.8, 0.9], 'min_child_weight': [2, 3, 4], 'scale_pos_weight': [scale_pos_weight]}
    lgb_params = {'n_estimators': [300, 500, 700], 'num_leaves': [15, 31], 'learning_rate': [0.01], 'max_depth': [4, 5, 6], 'subsample': [0.7, 0.8, 0.9], 'min_child_samples': [10, 20], 'scale_pos_weight': [scale_pos_weight]}
    rf_params = {'n_estimators': [300, 500, 700], 'max_depth': [5, 10], 'min_samples_split': [3, 4, 5], 'min_samples_leaf': [2, 3, 4], 'max_features': ['sqrt'], 'class_weight': ['balanced']}
    svm_params = {'C': [0.5, 1.0, 2.0], 'kernel': ['rbf', 'linear'], 'gamma': ['scale', 'auto']}
    
    def get_best_est(est, params):
        search = RandomizedSearchCV(est, params, n_iter=100, scoring='roc_auc', cv=5, n_jobs=-1, random_state=SEED, verbose=0)
        search.fit(X_train_final, y_train)
        return search.best_estimator_
    
    best_xgb = get_best_est(xgb.XGBClassifier(eval_metric='auc', random_state=SEED), xgb_params)
    best_lgbm = get_best_est(lgb.LGBMClassifier(objective='binary', verbosity=-1, random_state=SEED), lgb_params)
    best_rf = get_best_est(RandomForestClassifier(random_state=SEED), rf_params)
    best_svm = get_best_est(SVC(probability=True, random_state=SEED), svm_params)
    best_gnb = GaussianNB(var_smoothing=1e-9)
    
    stacking_model = StackingClassifier(
        estimators=[('xgb', best_xgb), ('lgbm', best_lgbm), ('rf', best_rf), ('svm', best_svm), ('gnb', best_gnb)],
        final_estimator=LogisticRegression(class_weight='balanced', random_state=SEED), cv=5, n_jobs=-1, passthrough=False
    )
    stacking_model.fit(X_train_final, y_train)
    
    # Find optimal threshold and evaluate
    val_probs = stacking_model.predict_proba(X_val_final)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)
    f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
    best_threshold = thresholds[np.argmax(f1_scores)]
    
    # Bootstrapping
    str_val, fpr_val, tpr_val, l_val, u_val = bootstrap_metrics_ci_with_roc(stacking_model, X_val_final, y_val, best_threshold)
    str_ext, fpr_ext, tpr_ext, l_ext, u_ext = bootstrap_metrics_ci_with_roc(stacking_model, X_ext_final, y_ext, best_threshold)

    print(f" [Internal Val] AUC: {str_val['AUC']} | F1: {str_val['F1']} | Acc: {str_val['Accuracy']} | Prec: {str_val['Precision']} | Rec: {str_val['Recall']}")
    print(f" [External Test] AUC: {str_ext['AUC']} | F1: {str_ext['F1']} | Acc: {str_ext['Accuracy']} | Prec: {str_ext['Precision']} | Rec: {str_ext['Recall']}")
    
    all_results[strategy] = {
        'val': {'str': str_val, 'fpr': fpr_val, 'tpr': tpr_val, 'lower': l_val, 'upper': u_val},
        'ext': {'str': str_ext, 'fpr': fpr_ext, 'tpr': tpr_ext, 'lower': l_ext, 'upper': u_ext}
    }

# ==========================================
# 5. ROC Plotting
# ==========================================
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['axes.linewidth'] = 1.2

fig, axes = plt.subplots(1, 2, figsize=(12, 6), dpi=300)

styles = {
    'ICU_Only': {'color': '#1f77b4', 'ls': '-', 'label': 'ICU 24h Baseline'},
    'Two_Stage': {'color': '#d62728', 'ls': '-', 'label': 'Two-Stage Proposed'}
}

# Iterate through axes: axes[0] = Internal, axes[1] = External
datasets_info = [('val', 'A. Internal Validation Cohort'), ('ext', 'B. External Test Cohort')]

for ax, (ds, title) in zip(axes, datasets_info):
    for strategy in strategies:
        res = all_results[strategy][ds]
        color = styles[strategy]['color']
        ls = styles[strategy]['ls']
        label_text = f"{styles[strategy]['label']} (AUC: {res['str']['AUC']})"
        
        ax.plot(res['fpr'], res['tpr'], color=color, lw=2, linestyle=ls, label=label_text)
        ax.fill_between(res['fpr'], res['lower'], res['upper'], color=color, alpha=0.15, linewidth=0)
    
    ax.plot([0, 1], [0, 1], linestyle=':', lw=1.5, color='gray', alpha=0.8)
    ax.set_xlim([-0.02, 1.02])
    ax.set_ylim([-0.02, 1.02])
    ax.set_xlabel('1 - Specificity (FPR)', fontsize=12, fontweight='medium')

    if ds == 'val':
        ax.set_ylabel('Sensitivity (TPR)', fontsize=12, fontweight='medium')
    
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)
    ax.legend(loc="lower right", fontsize=11, frameon=False)
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import copy
import matplotlib.pyplot as plt
from scipy.spatial import distance
from scipy.optimize import linear_sum_assignment
from sklearn.linear_model import LogisticRegression

SMD_THRESHOLD = 0.2
SEVERITY_VARS = ['MV', 'admissionroute', 'vasopressor', 'age']

def calculate_propensity_scores(X, treatment_col, covariate_list):
    """Calculate Propensity Score (PS) and Logit PS"""
    model = LogisticRegression(solver='liblinear', class_weight='balanced', random_state=42)
    model.fit(X[covariate_list], X[treatment_col])
    ps_score = model.predict_proba(X[covariate_list])[:, 1]
    
    X = X.copy()
    X['ps'] = ps_score
    X['ps'] = X['ps'].clip(1e-6, 1-1e-6)
    X['ps_logit'] = np.log(X['ps'] / (1 - X['ps']))
    return X

def get_mahalanobis_params(X, vars_list):
    """Precompute inverse covariance matrix for Mahalanobis distance"""
    cov_matrix = np.cov(X[vars_list].T)
    try:
        inv_cov_matrix = np.linalg.inv(cov_matrix)
    except np.linalg.LinAlgError:
        inv_cov_matrix = np.linalg.pinv(cov_matrix)
    return inv_cov_matrix

def check_balance(X, treatment_col, covariates):
    """Calculate Standardized Mean Difference (SMD)"""
    treated = X[X[treatment_col] == 1][covariates]
    control = X[X[treatment_col] == 0][covariates]
    
    if len(treated) == 0 or len(control) == 0:
        return pd.Series(index=covariates, dtype=float)
        
    mean_t = treated.mean()
    mean_c = control.mean()
    std_t = treated.std()
    std_c = control.std()
    
    pooled_std = np.sqrt((std_t**2 + std_c**2) / 2)
    pooled_std[pooled_std == 0] = 1e-9
    smd = (mean_t - mean_c).abs() / pooled_std
    return smd

# Core: Global Optimal Matching Algorithm
def perform_global_optimal_matching(df, treatment_col, covariates, severity_vars, caliper_scale=0.2):
    """
    Execute Global Optimal Matching:
    1. Calculate full distance matrix (Cost Matrix) between treatment and control groups.
    2. Distance = Mahalanobis Distance (Severity Vars) + Penalty (if PS Logit diff > caliper).
    3. Use Hungarian Algorithm (Linear Sum Assignment) to minimize total distance.
    """
    df_data = calculate_propensity_scores(df, treatment_col, covariates)
    treated_df = df_data[df_data[treatment_col] == 1]
    control_df = df_data[df_data[treatment_col] == 0]
    
    n_treated = len(treated_df)
    n_control = len(control_df)

    print(f"Starting global optimal matching... Treatment group: {n_treated}, Control group: {n_control}")
    std_logit = df_data['ps_logit'].std()
    caliper = caliper_scale * std_logit
    print(f"Setting caliper (Logit Scale): {caliper:.4f}")
    
    VI = get_mahalanobis_params(df_data, severity_vars)
    t_logits = treated_df['ps_logit'].values.reshape(-1, 1)
    c_logits = control_df['ps_logit'].values.reshape(1, -1)
    
    dist_logit = np.abs(t_logits - c_logits)
    t_feats = treated_df[severity_vars].values
    c_feats = control_df[severity_vars].values
    dist_mahal = distance.cdist(t_feats, c_feats, metric='mahalanobis', VI=VI)
    
    PENALTY = 1e6 
    cost_matrix = dist_mahal.copy()
    out_of_caliper_mask = dist_logit > caliper
    cost_matrix[out_of_caliper_mask] = PENALTY

    print("Cost matrix constructed, solving the optimal assignment problem (Hungarian Algorithm)...")
    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    
    valid_matches_t = []
    valid_matches_c = []
    for r, c in zip(row_ind, col_ind):
        if cost_matrix[r, c] < PENALTY:
            valid_matches_t.append(treated_df.index[r])
            valid_matches_c.append(control_df.index[c])
            
    matched_t_df = df_data.loc[valid_matches_t]
    matched_c_df = df_data.loc[valid_matches_c]
    df_matched = pd.concat([matched_t_df, matched_c_df])
    
    print("Matching complete.")
    print(f"Original treatment count: {n_treated} -> Successfully matched: {len(valid_matches_t)}")
    print(f"Match rate: {len(valid_matches_t)/n_treated*100:.2f}%")
    
    return df_matched

# Main Execution Flow
filename = '全部病房患者/外部测试集/preicu.csv'
df_raw = pd.read_csv(filename)
df = df_raw.copy()
df['T'] = df['lac'].notnull().astype(int)
print("Data loaded successfully.")

covariates = [
    'gender', 'age', 'admissionroute', 'vasopressor', 'MV',
    'temp_max', 'temp_min', 'resp_max', 'resp_min', 'hr_max', 'hr_min', 'map_max', 'map_min', 
    'tbil', 'glu', 'cre', 'plt', 'wbc', 'lym'
]

df_clean = df.dropna(subset=covariates).reset_index(drop=True)

print("\n--- Balance Pre-Matching ---")
pre_smd = check_balance(df_clean, 'T', covariates)
print(f"Variables with SMD > {SMD_THRESHOLD}: {(pre_smd > SMD_THRESHOLD).sum()}")
print(pre_smd)

df_matched = perform_global_optimal_matching(
    df_clean, 
    'T', 
    covariates, 
    severity_vars=SEVERITY_VARS, 
    caliper_scale=0.1
)

if not df_matched.empty:
    print("\n--- Balance Post-Matching ---")
    post_smd = check_balance(df_matched, 'T', covariates)
    print(f"Variables with SMD > {SMD_THRESHOLD}: {(post_smd > SMD_THRESHOLD).sum()}")
    print("SMD for major severity indicators:")
    print(post_smd)
    
    df_matched.to_csv('全部病房患者/外部测试集/rmyy_matched_optimal.csv', index=False)
    print("\nFile saved to rmyy_matched_optimal.csv")
    
    # Plot matching results
    plt.rcParams['font.sans-serif'] = ['Arial'] 
    plt.rcParams['axes.unicode_minus'] = False
    sns.set_context("paper", font_scale=1.2)
    sns.set_style("ticks")
    
    plt.figure(figsize=(8, 5), dpi=300)
    colors = ["#34495e", "#e74c3c"] 
    sns.histplot(
        data=df_matched, x='ps', hue='T', 
        stat="density",
        common_norm=False, 
        palette=colors,
        alpha=0.3,
        linewidth=1.5,
        kde=True
    )
    plt.title('Propensity Score Distribution After Matching', pad=15, fontsize=14, fontweight='bold')
    plt.xlabel('Propensity Score', fontsize=12)
    plt.ylabel('Probability Density', fontsize=12)
    plt.legend(
        title='Group', 
        labels=['Treated (Lac)', 'Control (No Lac)'],
        frameon=False,
        loc='upper right'
    )
    sns.despine()
    plt.tight_layout()
    plt.show()
else:
    print("Matching resulted in empty dataframe, caliper might be too strict or no overlap area exists.")

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import (RandomForestClassifier, GradientBoostingClassifier, StackingClassifier)
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.utils import resample
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, brier_score_loss)
from sklearn.calibration import calibration_curve
from scipy.stats import norm
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

# ==========================================
# 0. Global Setup
# ==========================================
sns.set_style("whitegrid")
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['axes.unicode_minus'] = False
warnings.filterwarnings("ignore")

SEED = 0
np.random.seed(SEED)

# ==========================================
# 1. Data Loading & Merging
# ==========================================
def load_and_merge_data(matched_file, icu_file):
    df_matched = pd.read_csv(matched_file)
    df_icu = pd.read_csv(icu_file)
    
    valid_ids = df_matched['pat_id'].unique()
    df_icu = df_icu[df_icu['pat_id'].isin(valid_ids)].copy()
    
    icu_cols = [c for c in df_icu.columns if c != 'pat_id']
    rename_dict = {c: f"{c}_s2" for c in icu_cols}
    df_icu = df_icu.rename(columns=rename_dict)
    
    df_merged = pd.merge(df_matched, df_icu, on='pat_id', how='inner')
    return df_merged

print(">>> Loading Development Data ...")
df_dev_all = load_and_merge_data('全部病房患者/内部训练集/rmyy_matched_optimal.csv', '全部病房患者/内部训练集/icu.csv')

print(">>> Loading External Test Data (23-25)...")
df_test_all = load_and_merge_data('全部病房患者/外部测试集/rmyy_matched_optimal.csv', '全部病房患者/外部测试集/icu.csv')

print(f"Dev Set Total: {len(df_dev_all)} | Ext Test Set Total: {len(df_test_all)}")

# ==========================================
# 2. Feature Definitions
# ==========================================
target_col = 'death_90d'

cols_stage1_base = [
    'gender', 'age', 'admissionroute', 'vasopressor', 'MV',
    'temp_max', 'temp_min', 'resp_max', 'resp_min', 'hr_max', 'hr_min', 'map_max', 'map_min',
    'tbil', 'glu', 'cre', 'plt', 'wbc', 'lym'
]

cols_stage2_base = [
    'gender_s2', 'age_s2', 'admissionroute_s2', 'vasopressor_s2', 'MV_s2',
    'temp_max_s2', 'temp_min_s2', 'resp_max_s2', 'resp_min_s2', 'hr_max_s2', 'hr_min_s2', 'map_max_s2', 'map_min_s2',
    'glu_s2', 'tbil_s2', 'cre_s2', 'spo2_s2', 'plt_s2', 'wbc_s2', 'lym_s2', 'ph_s2', 'lac_s2'
]

binary_cols_s1 = ['gender', 'vasopressor', 'MV', 'admissionroute']
binary_cols_s2 = ['gender_s2', 'vasopressor_s2', 'MV_s2', 'admissionroute_s2']

# ==========================================
# 3. Bootstrap & Statistical Testing
# ==========================================
def get_bootstrap_stats(model, X, y, threshold=0.5, n_bootstraps=1000):
    """Execute Bootstrap resampling, returns formatted strings and raw lists."""
    y_prob_full = model.predict_proba(X)[:, 1]
    raw_stats = {'AUC': [], 'Accuracy': [], 'Precision': [], 'Recall': [], 'F1': []}
    
    for i in range(n_bootstraps):
        indices = resample(np.arange(len(y)), random_state=i)
        if len(np.unique(y[indices])) < 2: 
            continue
            
        y_true = y[indices]
        y_prob = y_prob_full[indices]
        y_pred = (y_prob >= threshold).astype(int)
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        
        raw_stats['AUC'].append(auc(fpr, tpr))
        raw_stats['Accuracy'].append(accuracy_score(y_true, y_pred))
        raw_stats['Precision'].append(precision_score(y_true, y_pred, zero_division=0))
        raw_stats['Recall'].append(recall_score(y_true, y_pred, zero_division=0))
        raw_stats['F1'].append(f1_score(y_true, y_pred, zero_division=0))
        
    str_stats = {}
    for k, v in raw_stats.items():
        mean_v = np.mean(v)
        ci_lower = np.percentile(v, 2.5)
        ci_upper = np.percentile(v, 97.5)
        str_stats[k] = f"{mean_v:.3f} ({ci_lower:.3f}-{ci_upper:.3f})"
        
    return str_stats, raw_stats, y_prob_full

def calculate_p_value(dist_a, dist_b):
    """
    Compare two independent Bootstrap distributions, compute P-value (Z-test)
    H0: Mean_A == Mean_B
    H1: Mean_A != Mean_B (Two-tailed)
    """
    mean_a, std_a = np.mean(dist_a), np.std(dist_a, ddof=1)
    mean_b, std_b = np.mean(dist_b), np.std(dist_b, ddof=1)
    
    z_score = (mean_b - mean_a) / np.sqrt(std_a**2 + std_b**2)
    p_value = 2 * (1 - norm.cdf(abs(z_score)))
    return p_value, z_score

def calculate_net_benefit(y_true, y_prob, thresholds):
    """Calculate Net Benefit for DCA."""
    net_benefits = []
    n = len(y_true)
    for thresh in thresholds:
        pred = (y_prob >= thresh).astype(int)
        tp = np.sum((pred == 1) & (y_true == 1))
        fp = np.sum((pred == 1) & (y_true == 0))
        
        if thresh == 1.0:
            nb = 0
        else:
            nb = (tp / n) - (fp / n) * (thresh / (1 - thresh))
        net_benefits.append(nb)
    return np.array(net_benefits)

# ==========================================
# 4. Core Training Pipeline
# ==========================================
def run_two_stage_pipeline(df_dev, df_test, group_name, use_lac_in_s1):
    print(f"\n{'='*15} Processing: {group_name} {'='*15}")
    current_cols_s1 = cols_stage1_base.copy()
    
    if use_lac_in_s1:
        current_cols_s1.append('lac')
        print("  -> Stage 1: Included 'lac'")
    else:
        print("  -> Stage 1: No 'lac'")
        
    current_cols_s2 = cols_stage2_base.copy()
    
    X_dev = df_dev
    y_dev = df_dev[target_col].values
    train_idx, val_idx = train_test_split(
        np.arange(len(df_dev)), test_size=0.2, stratify=y_dev, random_state=SEED
    )
    
    X_ext = df_test
    y_ext = df_test[target_col].values
    
    print("  [Step 1] Training Pre-ICU Risk Score...")
    X_s1_train = X_dev.iloc[train_idx][current_cols_s1].reset_index(drop=True)
    y_train = y_dev[train_idx]
    X_s1_val = X_dev.iloc[val_idx][current_cols_s1].reset_index(drop=True)
    X_s1_ext = X_ext[current_cols_s1].reset_index(drop=True)
    
    cont_cols_s1 = [c for c in current_cols_s1 if c not in binary_cols_s1]
    bin_cols_s1_curr = [c for c in current_cols_s1 if c in binary_cols_s1]
    
    preprocessor_s1 = ColumnTransformer([
        ('num', RobustScaler(), cont_cols_s1),
        ('cat', 'passthrough', bin_cols_s1_curr)
    ])
    preprocessor_s1.set_output(transform='pandas')
    
    X_s1_train_sc = preprocessor_s1.fit_transform(X_s1_train)
    X_s1_val_sc = preprocessor_s1.transform(X_s1_val)
    X_s1_ext_sc = preprocessor_s1.transform(X_s1_ext)
    
    lr_s1 = LogisticRegression(class_weight='balanced', solver='liblinear', random_state=SEED)
    lr_s1.fit(X_s1_train_sc, y_train)
    risk_train = lr_s1.predict_proba(X_s1_train_sc)[:, 1]
    risk_val = lr_s1.predict_proba(X_s1_val_sc)[:, 1]
    risk_ext = lr_s1.predict_proba(X_s1_ext_sc)[:, 1]

    print("  [Step 2] Preparing Stacking Data...")
    X_s2_train = X_dev.iloc[train_idx][current_cols_s2].reset_index(drop=True)
    X_s2_val = X_dev.iloc[val_idx][current_cols_s2].reset_index(drop=True)
    X_s2_ext = X_ext[current_cols_s2].reset_index(drop=True)
    
    X_s2_train['pre_icu_risk'] = risk_train
    X_s2_val['pre_icu_risk'] = risk_val
    X_s2_ext['pre_icu_risk'] = risk_ext
    
    cont_cols_s2 = [c for c in current_cols_s2 if c not in binary_cols_s2] + ['pre_icu_risk']
    bin_cols_s2_curr = [c for c in current_cols_s2 if c in binary_cols_s2]
    
    preprocessor_s2 = ColumnTransformer([
        ('num', RobustScaler(), cont_cols_s2),
        ('cat', 'passthrough', bin_cols_s2_curr)
    ], verbose_feature_names_out=False)
    preprocessor_s2.set_output(transform='pandas')
    
    X_train_final = preprocessor_s2.fit_transform(X_s2_train)
    X_val_final = preprocessor_s2.transform(X_s2_val)
    X_ext_final = preprocessor_s2.transform(X_s2_ext)
    y_val = y_dev[val_idx]
    
    # --- 5. Stacking Training ---
    print("  [Step 3] Training & Optimizing Stacking Model...")
    pos_weight = np.sum(y_train==0) / np.sum(y_train==1)
    
    def optimize_model(est, params, name):
        print(f"    Optimizing {name}...")
        search = RandomizedSearchCV(est, params, n_iter=100, scoring='roc_auc', cv=5, n_jobs=-1, random_state=SEED, verbose=0)
        search.fit(X_train_final, y_train)
        return search.best_estimator_

    xgb_params = {
        'n_estimators': [300, 500, 700],
        'learning_rate': [0.01],
        'max_depth': [3, 4, 5],
        'subsample': [0.7, 0.8, 0.9],
        'colsample_bytree': [0.7, 0.8, 0.9],
        'min_child_weight': [2, 3, 4],
        'scale_pos_weight': [scale_pos_weight]
    }
    
    lgb_params = {
        'n_estimators': [300, 500, 700],
        'num_leaves': [15, 31],
        'learning_rate': [0.01],
        'max_depth': [4, 5, 6],
        'subsample': [0.7, 0.8, 0.9],
        'min_child_samples': [10, 20],
        'scale_pos_weight': [scale_pos_weight]
    }
    
    rf_params = {
        'n_estimators': [300, 500, 700],
        'max_depth': [5, 10],
        'min_samples_split': [3, 4, 5],
        'min_samples_leaf': [2, 3, 4],
        'max_features': ['sqrt'],
        'class_weight': ['balanced']
    }
    
    svm_params = {
        'C': [0.5, 1.0, 2.0],
        'kernel': ['rbf', 'linear'],
        'gamma': ['scale', 'auto']
    }

    best_xgb = optimize_model(xgb.XGBClassifier(eval_metric='auc', random_state=SEED, n_jobs=1), xgb_params, "XGBoost")
    best_lgbm = optimize_model(lgb.LGBMClassifier(objective='binary', metric='auc', verbosity=-1, random_state=SEED, n_jobs=1), lgb_params, "LightGBM")
    best_rf = optimize_model(RandomForestClassifier(random_state=SEED, n_jobs=-1), rf_params, "Random Forest")
    best_svm = optimize_model(SVC(probability=True, random_state=SEED), svm_params, "SVM")
    best_gnb = GaussianNB(var_smoothing=1e-9)
    
    stacking_clf = StackingClassifier(
        estimators=[('xgb', best_xgb), ('lgbm', best_lgbm), ('rf', best_rf), ('svm', best_svm), ('gnb', best_gnb)],
        final_estimator=LogisticRegression(C=1.0, random_state=SEED), cv=5, n_jobs=-1, passthrough=False
    )
    stacking_clf.fit(X_train_final, y_train)

    val_probs_temp = stacking_clf.predict_proba(X_val_final)[:, 1]
    p, r, t = precision_recall_curve(y_val, val_probs_temp)
    f1_curve = 2 * (p * r) / (p + r + 1e-8)
    best_thresh = t[np.argmax(f1_curve)]
    
    print(f"  -> Best Threshold found: {best_thresh:.4f}")
    
    print("  [Eval] Bootstrapping Internal Validation...")
    int_str, int_raw, _ = get_bootstrap_stats(stacking_clf, X_val_final, y_val, best_thresh)
    
    print("  [Eval] Bootstrapping External Test...")
    ext_str, ext_raw, ext_probs = get_bootstrap_stats(stacking_clf, X_ext_final, y_ext, best_thresh)
    
    return {
        'internal_str': int_str,
        'internal_raw': int_raw,
        'external_str': ext_str,
        'external_raw': ext_raw,
        'y_ext_true': y_ext,
        'y_ext_prob': ext_probs
    }

# ==========================================
# 5. Execute Experiments
# ==========================================
df_dev_0 = df_dev_all[df_dev_all['T'] == 0].copy()
df_dev_1 = df_dev_all[df_dev_all['T'] == 1].copy()
df_test_0 = df_test_all[df_test_all['T'] == 0].copy()
df_test_1 = df_test_all[df_test_all['T'] == 1].copy()

res_A = run_two_stage_pipeline(df_dev_0, df_test_0, "Model A (Control T=0)", use_lac_in_s1=False)
res_B = run_two_stage_pipeline(df_dev_1, df_test_1, "Model B (Treatment T=1)", use_lac_in_s1=True)

# ==========================================
# 6. Results Summary & Statistical Tests
# ==========================================
def print_stat_comparison(title, res_a, res_b, metric_key_str, metric_key_raw):
    print(f"\n>>> {title} <<<")
    auc_p, auc_z = calculate_p_value(res_a[metric_key_raw]['AUC'], res_b[metric_key_raw]['AUC'])
    
    print(f"{'Metric':<10} | {'Model A (Control)':<30} | {'Model B (Treatment)':<30} | {'P-Value (Z-test)':<20}")
    print("-" * 100)
    
    metrics = ['AUC', 'Accuracy', 'Precision', 'Recall', 'F1']
    for m in metrics:
        val_a = res_a[metric_key_str][m]
        val_b = res_b[metric_key_str][m]
        p_text = ""
        if m == 'AUC':
            sig_mark = "**" if auc_p < 0.05 else ""
            p_text = f"{auc_p:.4f} {sig_mark}"
        print(f"{m:<10} | {val_a:<30} | {val_b:<30} | {p_text:<20}")
        
    print("-" * 100)
    if auc_p < 0.05:
        print("Conclusion: Significant statistical difference in AUC between the two models (P < 0.05).")
    else:
        print("Conclusion: No statistically significant difference in AUC between the two models (P >= 0.05).")

print("\n" + "="*80)
print("FINAL STATISTICAL REPORT (Mean [95% CI])")
print("="*80)

print_stat_comparison("INTERNAL VALIDATION COMPARISON", res_A, res_B, 'internal_str', 'internal_raw')
print_stat_comparison("EXTERNAL TEST COMPARISON", res_A, res_B, 'external_str', 'external_raw')

# ==========================================
# 7. Visualization (External Test Set)
# ==========================================
y_true_a, y_prob_a = res_A['y_ext_true'], res_A['y_ext_prob']
y_true_b, y_prob_b = res_B['y_ext_true'], res_B['y_ext_prob']

fig, axes = plt.subplots(1, 3, figsize=(18, 6), dpi=120)

# --- A. ROC Curve ---
fpr_a, tpr_a, _ = roc_curve(y_true_a, y_prob_a)
auc_a = np.mean(res_A['external_raw']['AUC'])
fpr_b, tpr_b, _ = roc_curve(y_true_b, y_prob_b)
auc_b = np.mean(res_B['external_raw']['AUC'])

axes[0].plot(fpr_a, tpr_a, color='#1f77b4', lw=2, label=f'Model A (AUC={auc_a:.3f})')
axes[0].plot(fpr_b, tpr_b, color='#d62728', lw=2, label=f'Model B (AUC={auc_b:.3f})')
axes[0].plot([0, 1], [0, 1], linestyle=':', color='gray')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve (External Test)')
axes[0].legend(loc='lower right')
axes[0].grid(True, linestyle=':', alpha=0.6)

# --- B. Calibration Curve ---
prob_true_a, prob_pred_a = calibration_curve(y_true_a, y_prob_a, n_bins=10, strategy='uniform')
prob_true_b, prob_pred_b = calibration_curve(y_true_b, y_prob_b, n_bins=10, strategy='uniform')
brier_a = brier_score_loss(y_true_a, y_prob_a)
brier_b = brier_score_loss(y_true_b, y_prob_b)

axes[1].plot(prob_pred_a, prob_true_a, marker='o', color='#1f77b4', label=f'Model A (Brier={brier_a:.3f})')
axes[1].plot(prob_pred_b, prob_true_b, marker='s', color='#d62728', label=f'Model B (Brier={brier_b:.3f})')
axes[1].plot([0, 1], [0, 1], linestyle=':', color='gray', label='Perfectly Calibrated')
axes[1].set_xlabel('Mean Predicted Probability')
axes[1].set_ylabel('Fraction of Positives')
axes[1].set_title('Calibration Curve')
axes[1].legend(loc='best')
axes[1].grid(True, linestyle=':', alpha=0.6)

# --- C. Decision Curve Analysis (DCA) ---
thresholds = np.linspace(0.01, 0.99, 100)
nb_a = calculate_net_benefit(y_true_a, y_prob_a, thresholds)
nb_b = calculate_net_benefit(y_true_b, y_prob_b, thresholds)
prevalence = np.mean(y_true_a)
nb_all = prevalence - (1 - prevalence) * (thresholds / (1 - thresholds))

axes[2].plot(thresholds, nb_a, color='#1f77b4', lw=2, label='Model A')
axes[2].plot(thresholds, nb_b, color='#d62728', lw=2, label='Model B')
axes[2].plot(thresholds, nb_all, linestyle='--', color='gray', alpha=0.8, label='Treat All')
axes[2].axhline(y=0, color='black', linestyle=':', label='Treat None')

y_min = np.min([np.min(nb_a), np.min(nb_b), -0.1])
y_max = np.max([np.max(nb_a), np.max(nb_b), prevalence]) + 0.05
axes[2].set_ylim(max(y_min, -0.2), y_max)
axes[2].set_xlabel('Threshold Probability')
axes[2].set_ylabel('Net Benefit')
axes[2].set_title('Decision Curve Analysis (DCA)')
axes[2].legend(loc='upper right')
axes[2].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()